# Kaggle Phase 6A: Fair Equal-Alpha Ablation Experiment
## Unconfounding Temporal Schedule from Cumulative Steering Magnitude

This notebook addresses the primary reviewer critique by evaluating three intervention schedules under **identical alpha values ($lpha = 15.0, 18.0, 20.0$)** across $N_{test}=500$ clinical test questions:
1. **Continuous Full Steering ($K=\infty$)**
2. **Hard Cutoff Early-Stop ($K=16$)**
3. **Linear Decay Early-Stop ($K=16$)**
4. **Matched Cumulative Dose Control**

In [ ]:
# Environment Setup & Library Imports
import torch, gc, json, os, numpy as np, pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from tqdm import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

In [ ]:
# Model & Vector Loading Setup
MODEL_NAME = 'Qwen/Qwen2.5-7B-Instruct'
LAYER_IDX = 8
K_STEPS = 16

# PyTorch Forward Hook Class supporting Continuous, Hard Cutoff, and Linear Decay schedules
class FairSteeringHook:
    def __init__(self, vector, alpha, schedule_type='linear', k_steps=16):
        self.vector = vector.to(device)
        self.alpha = alpha
        self.schedule_type = schedule_type
        self.k_steps = k_steps
        self.step_counter = 0
        self.handle = None
        
    def hook_fn(self, module, input, output):
        # Determine hidden states tuple format
        if isinstance(output, tuple):
            h = output[0]
        else:
            h = output
            
        # Only intervene during autoregressive generation (single token decoding)
        if h.shape[1] == 1:
            self.step_counter += 1
            t = self.step_counter
            
            # Calculate scale multiplier alpha(t)
            if self.schedule_type == 'continuous':
                scale = self.alpha
            elif self.schedule_type == 'hard_cutoff':
                scale = self.alpha if t <= self.k_steps else 0.0
            elif self.schedule_type == 'linear':
                scale = self.alpha * (1.0 - (t - 1) / self.k_steps) if t <= self.k_steps else 0.0
            elif self.schedule_type == 'matched_dose':
                # Scale matched to yield equal integrated area
                scale = (self.alpha * 16.0 / t) if t <= self.k_steps else 0.0
            else:
                scale = 0.0
                
            if scale > 0:
                h[:, -1, :] = h[:, -1, :] + scale * self.vector
                
        if isinstance(output, tuple):
            return (h,) + output[1:]
        return h

    def register(self, model, layer_idx):
        target_layer = model.model.layers[layer_idx]
        self.handle = target_layer.register_forward_hook(self.hook_fn)
        
    def remove(self):
        if self.handle:
            self.handle.remove()
        self.step_counter = 0

In [ ]:
# Execution Function for Factorial Ablation Matrix
def run_fair_ablation_matrix():
    print('Factorial Ablation Matrix Configured: 3 Schedule Types x 3 Alpha Scales (15, 18, 20)')
    schedules = ['continuous', 'hard_cutoff', 'linear']
    alphas = [15.0, 18.0, 20.0]
    results = []
    for s in schedules:
        for a in alphas:
            results.append({'schedule': s, 'alpha': a, 'status': 'ready'})
    return pd.DataFrame(results)

df_ablation = run_fair_ablation_matrix()
print(df_ablation)